# 02 - Limpieza de datos

## Objetivo de este cuaderno

Aquí aplicamos, una por una, las decisiones tomadas en el EDA. Partimos siempre del CSV original, que nunca se modifica, y al final guardamos el resultado limpio.

- **Lee:** data/raw/coches_segunda_mano.csv
- **Guarda:** data/processed/coches_limpio.csv

### Importar librerías

Cargamos pandas, numpy, matplotlib y seaborn, que usaremos en todo el análisis.

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Cargar el dataset

Leemos el CSV de anuncios y mostramos las primeras filas para comprobar que la carga es correcta. Guardamos el número de filas para el resumen final.

In [2]:
df = pd.read_csv('../data/raw/coches_segunda_mano.csv')
filas_inicio = len(df)
print('Filas al cargar:', filas_inicio)
df.head()

Filas al cargar: 50000


,url,company,make,model,version,price,price_financed,fuel,year,kms,...,doors,shift,color,photos,is_professional,dealer,province,country,publish_date,insert_date
0,e158ae0ca53119ca199c28c36b5c2fcd,9881bcdd5a0ad4733037b3fb25e69c3a,SEAT,Toledo,SEAT Toledo 4p.,950,NaN,Diésel,2000.0,227000,...,4,Manual,Verde,5,False,0f4bb8455d27349b8273109b66a847f3,Navarra,Spain,2020-12-18 10:47:13,2021-01-15 00:00:00
1,ff267ebb7e700246f47f84f3db660b4b,9881bcdd5a0ad4733037b3fb25e69c3a,CITROEN,C1,CITROEN C1 PureTech 60KW 82CV Feel 5p.,6200,NaN,Gasolina,2017.0,50071,...,5,Manual,Blanco,6,True,Autos Raymara,Tenerife,Spain,2021-01-02 11:25:40,2021-01-15 00:00:00
2,de4b02db28ea7786c622b969be10c7c7,9881bcdd5a0ad4733037b3fb25e69c3a,FORD,Transit Connect,FORD Transit Connect Van 1.5 TDCi 100cv Ambien...,7851,7024.0,Diésel,2016.0,103000,...,4,Manual,Blanco,10,True,Auto 96,Barcelona,Spain,2020-12-16 10:51:45,2021-01-15 00:00:00
3,0449972a4d07594acf92e9a7dd28b39c,9881bcdd5a0ad4733037b3fb25e69c3a,VOLKSWAGEN,Caravelle,VOLKSWAGEN Caravelle Largo 2.0 TDI 140 Comfort...,19426,NaN,Diésel,2014.0,120000,...,4,Manual,Blanco,9,True,Inniauto,Navarra,Spain,2020-11-25 11:09:14,2021-01-15 00:00:00
4,12c4fa49bd4fdf23f19ecf396d3f02ef,9881bcdd5a0ad4733037b3fb25e69c3a,FORD,Transit,FORD Transit 350 96kW L4 Ambiente Propulsion T...,22850,22800.0,Diésel,2017.0,107000,...,2,Manual,Blanco,4,True,"Autofleet España,s.l",Sevilla,Spain,2021-01-12 20:00:34,2021-01-15 00:00:00


### Cortes del EDA

*Decisiones tomadas en el EDA (ver DECISIONES.md).*

- **Precio entre 1.000 € y 100.000 €:** por debajo la empresa no compra y por encima son casos aislados.
- **Hasta 1.000.000 km:** por encima no es un kilometraje creíble.
- **Potencia hasta 700 CV**, pero conservando los nulos: los nulos se tratan más adelante.
- **Coches de 2006 en adelante:** la empresa se centra en coches de 0 a 15 años. Este corte también elimina los 2 anuncios sin año, porque un nulo no cumple la condición.

Los aplicamos aquí, al principio, para no limpiar ni rellenar coches que luego se van a quitar.

In [3]:
df = df[(df['price'] >= 1000) & (df['price'] <= 100000) & (df['kms'] <= 1000000) & ((df['power'] <= 700) | (df['power'].isnull())) & (df['year'] >= 2006)]
filas_cortes = len(df)
print('Filas tras los cortes:', filas_cortes)
print('Filas eliminadas:', filas_inicio - filas_cortes)

Filas tras los cortes: 40801
Filas eliminadas: 9199


### Pasar year a entero

El año no tiene decimales; int16 basta y reduce memoria (int64 ocupa lo mismo que float64). Sus 2 nulos ya se eliminaron con el corte de 2006 en adelante; lo comprobamos antes de convertir.

In [4]:
df['year'].isnull().sum()

np.int64(0)

In [5]:
df['year'] = df['year'].astype('int16')
df['year'].dtype

dtype('int16')

### Eliminar columnas que no aportan

*Decisiones tomadas en el EDA (ver DECISIONES.md).*

- **price_financed:** falta en más de la mitad de los anuncios y, donde existe, va casi a la par que price (correlación ≈ 0,997), así que no aporta información nueva.
- **url, company y country:** url es un identificador encriptado, y company y country tienen el mismo valor en todos los anuncios.
- **dealer:** tiene casi 10.000 valores distintos, no aporta nada al modelo y quitarla mejora el anonimato.
- **color:** tiene miles de formas distintas de escribirse, influye poco en el precio de un coche usado y unificarla sería un trabajo desproporcionado.
- **insert_date:** es el mismo dato para todas las observaciones, no aporta nada al dataset de nuestro interés.

version no se quita todavía: la necesitamos para comprobar la potencia.

In [6]:
df = df.drop(columns=['price_financed', 'url', 'company', 'country', 'dealer', 'color', 'insert_date'])
df.shape

(40801, 14)

### Fechas

Pasamos publish_date (fecha de publicación del anuncio) a tipo fecha. pd.to_datetime convierte el texto en fecha y hora, y .dt.date se queda solo con el día, porque la hora no aporta nada.

In [7]:
df['publish_date'] = pd.to_datetime(df['publish_date']).dt.date
df['publish_date'].head()

1    2021-01-02
2    2020-12-16
3    2020-11-25
4    2021-01-12
5    2021-01-08
Name: publish_date, dtype: object

### Marca de los D2S

*Decisión tomada en el EDA (ver 01_eda.ipynb).*

Las dos filas sin marca tienen modelo D2S y no hay otro anuncio del que copiarla. Buscado fuera del dataset, corresponde a Invicta Electric (fuente: web oficial del fabricante, https://invictaelectric.es/tienda/d2s-17kw/, y ABC Motor, https://www.abc.es/motor/novedades/abci-invicta-coche-acceso-movilidad-electrica-202106170100_noticia.html). Escribimos la marca en mayúsculas, como el resto.

In [8]:
df.loc[df['make'].isnull() & (df['model'] == 'D2S'), 'make'] = 'INVICTA ELECTRIC'
df.loc[df['model'] == 'D2S', ['make', 'model', 'version']]

,make,model,version
5659,INVICTA ELECTRIC,D2S,D2S 27kW 3p.
47381,INVICTA ELECTRIC,D2S,D2S 27kW 3p.


### Comprobación: ¿tienen gemelos los coches sin potencia?

Tras los cortes, todavía hay anuncios sin potencia. Antes de decidir cómo rellenarla, medimos sobre los datos ya recortados si se puede copiar de **coches iguales** que sí la tienen:

1. Primero, coches con la misma marca, modelo y versión (la versión suele fijar el motor).
2. Si no hay, coches con la misma marca y modelo (menos preciso, porque un modelo puede tener varios motores).

En este bloque **solo medimos**: trabajamos con copias y df no cambia.

Separamos los anuncios sin potencia y con potencia.

In [9]:
sin_power = df[df['power'].isnull()].copy()
con_power = df[df['power'].notnull()].copy()
print('Sin potencia:', len(sin_power))
print('Con potencia:', len(con_power))

Sin potencia: 3842
Con potencia: 36959


Creamos una clave de texto con marca, modelo y versión, y otra solo con marca y modelo, para poder comparar.

In [10]:
sin_power['clave_mmv'] = sin_power['make'] + ' | ' + sin_power['model'] + ' | ' + sin_power['version']
con_power['clave_mmv'] = con_power['make'] + ' | ' + con_power['model'] + ' | ' + con_power['version']
sin_power['clave_mm'] = sin_power['make'] + ' | ' + sin_power['model']
con_power['clave_mm'] = con_power['make'] + ' | ' + con_power['model']

Contamos cuántos coches sin potencia tienen un gemelo exacto (marca + modelo + versión) con potencia.

In [11]:
gemelo_mmv = sin_power['clave_mmv'].isin(con_power['clave_mmv'])
print('Con gemelo marca + modelo + versión:', gemelo_mmv.sum())

Con gemelo marca + modelo + versión: 53


De los que no tienen gemelo exacto, contamos cuántos tienen al menos un coche de la misma marca y modelo con potencia.

In [12]:
restantes = sin_power[~gemelo_mmv]
gemelo_mm = restantes['clave_mm'].isin(con_power['clave_mm'])
print('Restantes sin gemelo exacto:', len(restantes))
print('De ellos, con gemelo marca + modelo:', gemelo_mm.sum())
print('Sin ningún gemelo:', len(restantes) - gemelo_mm.sum())

Restantes sin gemelo exacto: 3789
De ellos, con gemelo marca + modelo: 3729
Sin ningún gemelo: 60


### Observaciones (gemelos de los coches sin potencia)

- Tras los cortes quedan **3.842** anuncios sin potencia.
- Solo **53** tienen un gemelo exacto con la misma marca, modelo y versión.
- El motivo se ve en los ejemplos: en muchos de estos anuncios la versión no describe el motor. A menudo es solo la marca ("BMW", "VOLKSWAGEN"...), así que casi nunca coincide con una versión completa.
- De los 3.789 restantes, **3.729** sí tienen coches de la misma marca y modelo con potencia. Solo **60** no tienen ningún gemelo.
- Ojo: un mismo modelo puede llevar motores muy distintos, así que copiar la potencia de la misma marca y modelo es una aproximación, no un dato exacto.
- Conclusión: la versión casi nunca sirve para encontrar gemelos; el nivel que sí se puede usar es la marca y el modelo.


### Potencia escrita en la versión

Algunas versiones incluyen la potencia en el texto (por ejemplo, "FIAT 500 1.2 8v 51kW 69CV Lounge 3p."). Comprobamos cuántos coches sin potencia la tienen escrita en su versión, para usar ese dato en lugar de estimarlo.

Para contarlos usamos str.contains: busca un texto dentro de cada valor de la columna version y devuelve True o False. Buscamos "cv", "kw" o "caballos", sin distinguir mayúsculas (case=False). Con na=False, las versiones vacías cuentan como False en lugar de dar error. Sumando los True obtenemos cuántos coches cumplen la condición.

In [14]:
# Coches sin potencia
sin_potencia = df[df['power'].isnull()]

# De esos, cuántos tienen "cv", "kw" o "caballos" escrito en la versión
contiene_potencia = sin_potencia['version'].str.contains('cv|kw|caballos', case=False, na=False)

print('Coches sin potencia:', len(sin_potencia))
print('Con potencia escrita en la versión:', contiene_potencia.sum())

# Para ver cuáles son
sin_potencia[contiene_potencia][['make', 'model', 'version']]

Coches sin potencia: 3842
Con potencia escrita en la versión: 17


,make,model,version
609,FIAT,500,FIAT 500 1.2 8v 51kW 69CV Lounge 3p.
5142,FIAT,500,FIAT 500 1.2 8v 51kW 69CV Lounge 3p.
7164,CITROEN,Jumpy,CITROEN Jumpy Dob.Cab.Abat. M BlueHDi 90kW Con...
11659,SKODA,Kodiaq,SKODA Kodiaq 1.5 TSI 110KW 150cv DSG 4x2 Ambit...
12411,FIAT,Talento,FIAT Talento 1.2 SX Largo 1.6 EcoJet 89 kW 120 CV
13055,SKODA,Octavia,SKODA Octavia Combi 1.0 TSI 85KW 115CV Ambitio...
21013,VOLKSWAGEN,Touran,VOLKSWAGEN Touran Business 1.2 TSI 81kW 110CV 5p.
25852,KIA,Sportage,KIA Sportage 1.6 CRDi 100kW 136CV Drive 4x2 5p.
25958,FIAT,500,FIAT 500 Lounge 1.2 8v 51KW 69 CV 3p.
29311,KIA,Sportage,KIA Sportage 1.6 CRDi 100kW 136CV Drive 4x2 5p.


Son 17 coches, con 12 textos de versión distintos (el FIAT 500 aparece escrito de dos formas).

Cuando la versión da los CV, usamos ese valor. Cuando solo da los kW, los pasamos a CV multiplicando por 1,36 y redondeando: Citroën Jumpy 90 kW ≈ 122 CV y Ford Focus 92 kW ≈ 125 CV. Solo cambiamos filas sin potencia (power vacío), con df.loc, una versión por línea.

In [15]:
nulos_power_antes = df['power'].isnull().sum()

df.loc[df['power'].isnull() & (df['version'] == 'FIAT 500 1.2 8v 51kW 69CV Lounge 3p.'), 'power'] = 69
df.loc[df['power'].isnull() & (df['version'] == 'FIAT 500 Lounge 1.2 8v 51KW 69 CV 3p.'), 'power'] = 69
df.loc[df['power'].isnull() & (df['version'] == 'CITROEN Jumpy Dob.Cab.Abat. M BlueHDi 90kW Confort'), 'power'] = 122
df.loc[df['power'].isnull() & (df['version'] == 'SKODA Kodiaq 1.5 TSI 110KW 150cv DSG 4x2 Ambition 5p.'), 'power'] = 150
df.loc[df['power'].isnull() & (df['version'] == 'FIAT Talento 1.2 SX Largo 1.6 EcoJet 89 kW 120 CV'), 'power'] = 120
df.loc[df['power'].isnull() & (df['version'] == 'SKODA Octavia Combi 1.0 TSI 85KW 115CV Ambition 5p.'), 'power'] = 115
df.loc[df['power'].isnull() & (df['version'] == 'VOLKSWAGEN Touran Business 1.2 TSI 81kW 110CV 5p.'), 'power'] = 110
df.loc[df['power'].isnull() & (df['version'] == 'KIA Sportage 1.6 CRDi 100kW 136CV Drive 4x2 5p.'), 'power'] = 136
df.loc[df['power'].isnull() & (df['version'] == 'SKODA Kamiq 1.0 TSI 85kW 115CV DSG Style 5p.'), 'power'] = 115
df.loc[df['power'].isnull() & (df['version'] == 'FORD Focus 1.0 Ecoboost 92kW Trend 5p.'), 'power'] = 125
df.loc[df['power'].isnull() & (df['version'] == 'MAZDA Mazda2 1.5 GE 66kW 90CV Black Tech Edition 5p.'), 'power'] = 90
df.loc[df['power'].isnull() & (df['version'] == 'SKODA Scala SCALA 1.6 TDI 85 KW 115 CV Ambition 5p.'), 'power'] = 115

print('Nulos de power antes:', nulos_power_antes)
print('Nulos de power después:', df['power'].isnull().sum())
print('Rellenados desde la versión:', nulos_power_antes - df['power'].isnull().sum())

Nulos de power antes: 3842
Nulos de power después: 3825
Rellenados desde la versión: 17


### Rellenar la potencia con la moda de marca + modelo

*Decisión tomada en el EDA (ver DECISIONES.md).*

Para el resto de coches sin potencia usamos la **moda** (el valor más repetido) de los coches de la misma marca y modelo. No usamos la media ni la mediana porque podrían dar potencias que no existen en ese modelo; la moda es un motor real y frecuente.

Primero contamos cuántas veces aparece cada potencia en cada marca y modelo (los coches sin potencia no se cuentan).

In [16]:
conteo = df.groupby(['make', 'model', 'power']).size().reset_index(name='veces')

In [17]:
conteo

,make,model,power,veces
0,ABARTH,124 Spider,170.0,2
1,ABARTH,500,135.0,9
2,ABARTH,500,140.0,6
3,ABARTH,500,145.0,12
4,ABARTH,500,160.0,7
...,...,...,...,...
3482,VOLVO,XC90,250.0,3
3483,VOLVO,XC90,310.0,1
3484,VOLVO,XC90,320.0,3
3485,VOLVO,XC90,390.0,1


Ordenamos de más a menos repetida y, dentro de cada número de veces, de menor a mayor potencia; nos quedamos con la primera fila de cada marca y modelo, que es su moda. Así, si dos potencias empatan, gana la más baja y el resultado es siempre el mismo. Además es la opción prudente para el negocio: una potencia más baja da un precio estimado algo menor.

In [18]:
conteo = conteo.sort_values(['veces', 'power'], ascending=[False, True])
modas = conteo.drop_duplicates(subset=['make', 'model'])[['make', 'model', 'power']]
modas = modas.rename(columns={'power': 'power_moda'})

Unimos la moda a cada coche por marca y modelo, rellenamos con ella los nulos de power y quitamos la columna auxiliar.

In [20]:
df = df.merge(modas, on=['make', 'model'], how='left')
df['power'] = df['power'].fillna(df['power_moda'])
df = df.drop(columns=['power_moda'])
print('Nulos de power tras la moda:', df['power'].isnull().sum())

Nulos de power tras la moda: 60


### Eliminar los coches sin potencia ni gemelos

*Decisión tomada en el EDA (ver DECISIONES.md).*

Los coches que siguen sin potencia no tienen ningún coche de la misma marca y modelo con potencia. Los eliminamos en lugar de rellenarlos con la media general, que daría potencias que no existen y mezclaría segmentos distintos.

In [21]:
filas_antes = len(df)
df = df.dropna(subset=['power'])
filas_power = len(df)
print('Filas eliminadas sin potencia:', filas_antes - filas_power)

Filas eliminadas sin potencia: 60


### Pasar power a entero

La potencia no tiene decimales, así que la pasamos a int16. Solo se puede hacer ahora, después de rellenar y quitar los coches sin potencia, porque int16 no admite nulos.

In [22]:
df['power'] = df['power'].astype('int16')
df['power'].dtype

dtype('int16')

### Eliminar las filas con nulos pequeños

*Decisión tomada en el EDA (ver DECISIONES.md).*

Quitamos las filas sin combustible (fuel), cambio (shift), provincia (province) o modelo (model). Son muy pocas (menos del 0,5 % de los anuncios tras los cortes) y rellenarlas no compensa. Primero miramos cuántas hay de cada una.

In [23]:
df[['fuel', 'shift', 'province', 'model']].isnull().sum()

fuel        42
shift       81
province     4
model        0
dtype: int64

In [24]:
df = df.dropna(subset=['fuel', 'shift', 'province', 'model'])
filas_nulos = len(df)
print('Filas eliminadas por nulos pequeños:', filas_power - filas_nulos)

Filas eliminadas por nulos pequeños: 127


### Eliminar la columna version

*Decisión tomada en el EDA (ver DECISIONES.md).*

Repite la marca y el modelo, y su parte útil, la potencia, ya está en la columna power. El resto del texto no es uniforme. Ya la hemos usado para comprobar la potencia, así que ahora la quitamos.

In [25]:
df = df.drop(columns='version')

### Comprobación final

Tamaño final, nulos por columna (deberían ser todos 0) y resumen de las filas que quita cada paso.

In [26]:
df.shape

(40614, 13)

In [27]:
df.isnull().sum()

make               0
model              0
price              0
fuel               0
year               0
kms                0
power              0
doors              0
shift              0
photos             0
is_professional    0
province           0
publish_date       0
dtype: int64

In [28]:
print('Filas al cargar:', filas_inicio)
print('Quitadas por los cortes:', filas_inicio - filas_cortes)
print('Quitadas por no tener potencia ni gemelos:', filas_cortes - filas_power)
print('Quitadas por nulos pequeños:', filas_power - filas_nulos)
print('Filas finales:', len(df))
print('Total quitadas:', filas_inicio - len(df), '(', round((filas_inicio - len(df)) / filas_inicio * 100, 1), '% )')

Filas al cargar: 50000
Quitadas por los cortes: 9199
Quitadas por no tener potencia ni gemelos: 60
Quitadas por nulos pequeños: 127
Filas finales: 40614
Total quitadas: 9386 ( 18.8 % )


### Guardar el dataset limpio

Guardamos el resultado en data/processed/coches_limpio.csv, sin el índice. Al guardarlo en CSV, publish_date se escribe como texto, así que al volver a leerlo hay que convertirla de nuevo con pd.to_datetime.

In [29]:
df.to_csv('../data/processed/coches_limpio.csv', index=False)
print('Guardado:', df.shape)

Guardado: (40614, 13)
